# Overview and setup

## The app you'll build

- Every time you use a coding agent like Claude Code, Codex, or Copilot, it stores metadata about every session on your laptop.
    - The logs live in places like `~/.claude/projects/` as JSONL files, one JSON object per line.
    - They contain usage data, token counts, model names, tool calls - valuable data trapped in an awkward nested format.
- In this workshop, taught by Alena Astrakhantseva from dltHub, we turn those logs into structured tables and dashboards.
    - We do it with dlt and the dltHub AI workbench, which lets a coding agent build pipelines from natural-language prompts.
- By the end you'll have:
    1. A dlt pipeline loading local Claude Code logs into DuckDB.
    2. A marimo dashboard over that data with activity, models, tokens, and projects.
    3. A REST API pipeline pulling agent traces from a hosted API.
    4. A scheduled deployment on the dltHub Platform with a shareable dashboard.
- The architecture looks like this:

```mermaid
flowchart LR
    A[Sources] -->|dlt pipelines| B[(DuckDB)]
    B --> C[marimo dashboards]
    B --> D[dltHub Platform]
    subgraph A [Sources]
        A1[Local JSONL logs]
        A2[REST API traces]
    end
```

## Prerequisites

You'll need these accounts and tools:
- Python 3.11 or later
- [uv](https://docs.astral.sh/uv/) package manager
- A coding agent: Claude Code, Codex, or Copilot
- A dltHub Platform account (free): [app.dlthub.com](https://app.dlthub.com/)
- Some local agent logs so `~/.claude/projects/` has JSONL files to load, and if you don't have any yet, use your agent for a bit and come back.
    - To verify you have jsonl files in the claude projects directory, run the follwing command:

In [ ]:
# %%bash
# find ~/.claude/projects -type f -name "*.jsonl" | head -5

## Scaffold the workspace


- The dltHub AI workbench has its own scaffolding command, so run it in an empty folder:
- This creates a workspace with `pyproject.toml`, a `.dlt/` config directory, `.claude/` skills, and `.mcp.json` for the MCP server.
- It also creates `__deployment__.py` for cloud deployment and a virtual environment.
- When it asks to create a virtual environment and install dependencies, say yes. It runs `uv sync` for you.

In [ ]:
# %%bash
# uvx dlthub-init@latest

## Open the workspace in your agent

- Open the scaffolded folder in your coding agent.
- The agent reads the router skill and dispatches to the right toolkit when you ask it to build a pipeline.

In [ ]:
# %%bash
# uv run --active dlthub ai init

- Confirm the workbench is running

In [ ]:
# %%bash
# uv run --active dlthub ai status

In [ ]:
# %%bash
# uv run --active dlthub ai toolkit list

- DuckDB is our destination for local development.
    - It's an in-process analytical database - no server to run
    - dlt writes to a `.duckdb` file on disk. No extra setup is needed because DuckDB comes as a dependency of dlt.

# Part 1: Local logs to pipeline

- With the workspace scaffolded, we now build a dlt pipeline that reads the JSONL session transcripts from `~/.claude/projects/` and loads them into DuckDB.
- We don't write the code by hand. We tell the agent what to build, and it uses the dltHub AI workbench to write the pipeline.

## Look at the raw logs

- Open `~/.claude/projects/` and pick a `.jsonl` file. Every session is one file with one JSON object per line.
- The `type` values vary across lines. Some common ones:
    - `user`
    - `assistant`
    - `attachment`
    - `file-history-snapshot`
- The data is deeply nested, with usage objects holding token counts and message objects holding content arrays.
- This is real, valuable data - models used, tokens spent, and tools called. But it's trapped in a format that's painful to query manually.

## Build the pipeline

- Tell the agent to build a dlt pipeline for the local logs:
    > build a dlt pipeline, load data from local Claude logs as raw JSONs into DuckDB
- The agent starts with the dltHub router skill, which figures out that the data lives in files on disk.
- It installs the filesystem-pipeline toolkit on demand
    - this toolkit didn't exist in the project when you started.
    - The router pulls it in based on the data source.
- The toolkit walks the agent through the standard workflow:
    - confirm the plan
    - scaffold the pipeline
    - configure credentials
    - run it

## The pipeline the agent builds

- The pipeline uses dlt's `filesystem` source with the `read_jsonl` reader.
- The source lists files matching a glob, and the reader opens each one and yields parsed JSON records.
- dlt connects them with the pipe operator (`|`)
- The full pipeline (see [filesystem_pipeline.py](filesystem_pipeline.py)) defines a `load` function that creates the pipeline and runs it
- A few things to notice.
    - `dev_mode=True` adds a timestamp to the dataset name on every run, so each run starts fresh.
        - That's convenient during development but wasteful in production - we'll switch it off later.
    - `write_disposition="replace"` drops and reloads the table each time.

## Normalization

- When the pipeline runs, dlt doesn't just dump the raw JSON into one table.
    - It normalizes the data by inferring types, flattening nested objects, and creating child tables for nested arrays, linked by `_dlt_id` and `_dlt_parent_id`.

## View the data locally

- dlt ships with a built-in dashboard. Run it from the command line, not from the agent session.
- Make sure your pipeline ran successfully first:
    - run this at the same level of the nested uv `pyproject.toml` file for this sub-project not the root uv file
    ```bash
    uv run --active dlthub local show
    ```
- This opens a marimo dashboard that reads from the local DuckDB file.
- You can browse the schema, see how many tables exist, look at the data in each table, and run SQL queries.
- This is where you validate that the pipeline loaded what you expected.